In [1]:
import os, gc, math, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader, SequentialSampler
from transformers import AutoTokenizer, AutoModel, AutoConfig
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


torch: 2.10.0+cu128
CUDA available: True


In [2]:
MODEL_NAME = "microsoft/codebert-base"
MAX_LENGTH = 512
NUM_CLASSES = 6
DROPOUT = 0.30
BATCH_SIZE = 8
FOCAL_GAMMA = 2.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

LABEL_NAMES = {
    0: "Async Wait", 1: "Concurrency", 2: "Time",
    3: "Unordered Collections", 4: "Test Order Dependency", 5: "Non-flaky",
}

Device: cuda


## CodeBERT Architecture

In [3]:
 class BERT_Arch(nn.Module):
    def __init__(self, auto_model, num_classes=NUM_CLASSES):
        super().__init__()
        self.bert = auto_model
        self.dropout = nn.Dropout(DROPOUT)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(768, 512)
        self.fc2 = nn.Linear(512, num_classes)
        self.log_softmax = nn.LogSoftmax(dim=-1)

    def forward(self, input_ids, attention_mask):
        input_ids = input_ids.long()
        attention_mask = attention_mask.long()
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        pooled = outputs[1]
        x = self.fc1(pooled)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return self.log_softmax(x)


def build_fresh_model():
    model_config = AutoConfig.from_pretrained(MODEL_NAME, return_dict=False, output_hidden_states=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    auto_model = AutoModel.from_pretrained(MODEL_NAME, config=model_config)
    model = BERT_Arch(auto_model, NUM_CLASSES).to(DEVICE)
    return model, tokenizer

In [4]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, model_output, targets):
        ce_loss = F.cross_entropy(model_output, targets, reduction="none", weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == "mean" else focal_loss.sum()


def compute_class_balanced_focal_loss(train_labels):
    class_weights_np = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_CLASSES), y=train_labels)
    weights = torch.tensor(class_weights_np, dtype=torch.float32, device=DEVICE)
    return FocalLoss(alpha=weights, gamma=FOCAL_GAMMA), weights

In [5]:
def tokenize_texts(tokenizer, texts):
    return tokenizer(texts.tolist(), max_length=MAX_LENGTH, padding="max_length", truncation=True, return_tensors="pt")

def make_loader(tokens, labels, batch_size, shuffle):
    dataset = TensorDataset(tokens["input_ids"], tokens["attention_mask"], torch.tensor(labels, dtype=torch.long))
    return DataLoader(dataset, sampler=SequentialSampler(dataset), batch_size=batch_size)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    preds_all, labels_all = [], []
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        preds_all.append(torch.argmax(outputs, dim=1).cpu().numpy())
        labels_all.append(labels.cpu().numpy())
    return total_loss / max(1, len(loader)), np.concatenate(preds_all), np.concatenate(labels_all)


## Load pretrained fold checkpoints

In [9]:
def load_fold_model_from_checkpoint(checkpoint_path):
    model, tokenizer = build_fresh_model()
    state_dict = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model, tokenizer

def evaluate_fold_from_checkpoint(fold_idx, checkpoint_path, train_df, test_df):
    print(f"\n{'='*70}\nFOLD {fold_idx} -- evaluating checkpoint (no training)\n{'='*70}")
    model, tokenizer = load_fold_model_from_checkpoint(checkpoint_path)
    tokens_test = tokenize_texts(tokenizer, test_df["full_code"])
    test_loader = make_loader(tokens_test, test_df["category"].to_numpy(), BATCH_SIZE, shuffle=False)
    criterion, _ = compute_class_balanced_focal_loss(train_df["category"].to_numpy())
    test_loss, test_preds, test_labels = evaluate(model, test_loader, criterion, DEVICE)
    report = classification_report(
        test_labels, test_preds, labels=list(range(NUM_CLASSES)),
        target_names=[LABEL_NAMES[i] for i in range(NUM_CLASSES)], output_dict=True, zero_division=0,
    )
    print("Fold", fold_idx, "test macro-F1:", report["macro avg"]["f1-score"])
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return {
        "fold": fold_idx, "report": report, "test_preds": test_preds, "test_labels": test_labels,
        "confusion_matrix": confusion_matrix(test_labels, test_preds, labels=list(range(NUM_CLASSES))),
    }

In [10]:
EXACT_SPLITS_DIR = "/kaggle/input/datasets/zaimast7454/exact-splits/exact_splits"      
PRETRAINED_WEIGHTS_DIR = "/kaggle/input/datasets/zaimast7454/projectfolds"  
PRETRAINED_FILENAME_TEMPLATE = "per_project_model_weights_on__dataset_project_group_{fold}.pt"  
def load_exact_fold_split(fold_idx, splits_dir):
    splits_dir = Path(splits_dir)
    train_df = pd.read_csv(splits_dir / f"train_set_{fold_idx}.csv")
    test_df  = pd.read_csv(splits_dir / f"test_set_{fold_idx}.csv")
    for d in (train_df, test_df):
        d["project"] = d["project"].astype(str)
        d["full_code"] = d["full_code"].fillna("").astype(str)
        d["category"] = pd.to_numeric(d["category"], errors="raise").astype(int)
    assert not (set(train_df["project"]) & set(test_df["project"])), f"fold {fold_idx}: leakage!"
    return train_df, test_df

fold_results = []
for i in range(1, 5):
    train_df, test_df = load_exact_fold_split(i, EXACT_SPLITS_DIR)
    ckpt_path = Path(PRETRAINED_WEIGHTS_DIR) / PRETRAINED_FILENAME_TEMPLATE.format(fold=i)
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    fold_results.append(evaluate_fold_from_checkpoint(i, ckpt_path, train_df, test_df))

print(f"\n{len(fold_results)}/4 folds evaluated.")


FOLD 1 -- evaluating checkpoint (no training)


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fold 2 test macro-F1: 0.906832298136646

FOLD 3 -- evaluating checkpoint (no training)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fold 3 test macro-F1: 0.9582905982905984

FOLD 4 -- evaluating checkpoint (no training)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fold 4 test macro-F1: 0.9505868574922284

4/4 folds evaluated.


In [11]:
def summarize_folds(fold_results):
    rows = []
    for r in fold_results:
        row = {"fold": r["fold"], **{LABEL_NAMES[i]: r["report"][LABEL_NAMES[i]]["f1-score"]*100 for i in range(NUM_CLASSES)}}
        row["Macro Avg"] = r["report"]["macro avg"]["f1-score"] * 100
        rows.append(row)
    per_fold = pd.DataFrame(rows).set_index("fold")

    all_preds = np.concatenate([r["test_preds"] for r in fold_results])
    all_labels = np.concatenate([r["test_labels"] for r in fold_results])
    pooled = classification_report(all_labels, all_preds, labels=list(range(NUM_CLASSES)),
                                    target_names=[LABEL_NAMES[i] for i in range(NUM_CLASSES)],
                                    output_dict=True, zero_division=0)
    overall = {LABEL_NAMES[i]: pooled[LABEL_NAMES[i]]["f1-score"]*100 for i in range(NUM_CLASSES)}
    overall["Macro Avg"] = pooled["macro avg"]["f1-score"] * 100
    return per_fold, pd.DataFrame([overall], index=["Overall (pooled)"])

per_fold_table, overall_table = summarize_folds(fold_results)
print("Per-fold F1 (%):"); display(per_fold_table.round(2))
print("\nPooled -- comparable to paper Table 2:"); display(overall_table.round(2))

Per-fold F1 (%):


,Async Wait,Concurrency,Time,Unordered Collections,Test Order Dependency,Non-flaky,Macro Avg
fold,,,,,,,
1,83.12,80.00,66.67,82.35,57.89,100.0,78.34
2,86.96,57.14,100.00,100.00,100.00,100.0,90.68
3,96.00,100.00,93.33,92.31,93.33,100.0,95.83
4,88.24,90.91,100.00,95.65,95.56,100.0,95.06



Pooled -- comparable to paper Table 2:


,Async Wait,Concurrency,Time,Unordered Collections,Test Order Dependency,Non-flaky,Macro Avg
Overall (pooled),86.79,83.33,86.49,90.48,87.72,100.0,89.13


In [12]:
EXACT_SPLITS_DIR = "/kaggle/input/datasets/zaimast7454/exact-splits/exact_splits"       
PRETRAINED_WEIGHTS_DIR = "/kaggle/input/datasets/zaimast7454/projectfolds" 
PRETRAINED_FILENAME_TEMPLATE = "per_project_model_weights_on__dataset_project_group_{fold}.pt"  

def load_exact_fold_split(fold_idx, splits_dir):
    splits_dir = Path(splits_dir)
    test_df = pd.read_csv(splits_dir / f"test_set_{fold_idx}.csv")
    test_df["project"] = test_df["project"].astype(str)
    test_df["full_code"] = test_df["full_code"].fillna("").astype(str)
    test_df["category"] = pd.to_numeric(test_df["category"], errors="raise").astype(int)
    return test_df

def evaluate_fold_from_checkpoint(fold_idx, checkpoint_path, test_df):
    print(f"\n{'='*70}\nFOLD {fold_idx} -- evaluating checkpoint (no training)\n{'='*70}")
    model, tokenizer = load_fold_model_from_checkpoint(checkpoint_path)
    tokens_test = tokenize_texts(tokenizer, test_df["full_code"])
    test_loader = make_loader(tokens_test, test_df["category"].to_numpy(), BATCH_SIZE, shuffle=False)

    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in test_loader:
            input_ids, attention_mask = input_ids.to(DEVICE), attention_mask.to(DEVICE)
            outputs = model(input_ids, attention_mask)
            preds_all.append(torch.argmax(outputs, dim=1).cpu().numpy())
            labels_all.append(labels.numpy())
    test_preds = np.concatenate(preds_all)
    test_labels = np.concatenate(labels_all)

    report = classification_report(
        test_labels, test_preds, labels=list(range(NUM_CLASSES)),
        target_names=[LABEL_NAMES[i] for i in range(NUM_CLASSES)], output_dict=True, zero_division=0,
    )
    print("Fold", fold_idx, "test macro-F1:", report["macro avg"]["f1-score"])
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return {
        "fold": fold_idx, "report": report, "test_preds": test_preds, "test_labels": test_labels,
        "confusion_matrix": confusion_matrix(test_labels, test_preds, labels=list(range(NUM_CLASSES))),
    }

fold_results = []
for i in range(1, 5):
    test_df = load_exact_fold_split(i, EXACT_SPLITS_DIR)
    ckpt_path = Path(PRETRAINED_WEIGHTS_DIR) / PRETRAINED_FILENAME_TEMPLATE.format(fold=i)
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    fold_results.append(evaluate_fold_from_checkpoint(i, ckpt_path, test_df))

print(f"\n{len(fold_results)}/4 folds evaluated.")


FOLD 1 -- evaluating checkpoint (no training)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fold 1 test macro-F1: 0.7833853796702094

FOLD 2 -- evaluating checkpoint (no training)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fold 2 test macro-F1: 0.906832298136646

FOLD 3 -- evaluating checkpoint (no training)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fold 3 test macro-F1: 0.9582905982905984

FOLD 4 -- evaluating checkpoint (no training)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fold 4 test macro-F1: 0.9505868574922284

4/4 folds evaluated.


In [14]:
print("Cross-fold contamination check: does checkpoint i secretly perform well on OTHER folds' test sets too?\n")

for ckpt_i in range(1, 5):
    ckpt_path = Path(PRETRAINED_WEIGHTS_DIR) / PRETRAINED_FILENAME_TEMPLATE.format(fold=ckpt_i)
    model, tokenizer = load_fold_model_from_checkpoint(ckpt_path)

    print(f"--- Checkpoint {ckpt_i} ---")
    for test_j in range(1, 5):
        test_df_j = load_exact_fold_split(test_j, EXACT_SPLITS_DIR)   
        tokens = tokenize_texts(tokenizer, test_df_j["full_code"])
        loader = make_loader(tokens, test_df_j["category"].to_numpy(), BATCH_SIZE, shuffle=False)

        preds_all, labels_all = [], []
        model.eval()
        with torch.no_grad():
            for input_ids, attention_mask, labels in loader:
                input_ids, attention_mask = input_ids.to(DEVICE), attention_mask.to(DEVICE)
                outputs = model(input_ids, attention_mask)
                preds_all.append(torch.argmax(outputs, dim=1).cpu().numpy())
                labels_all.append(labels.numpy())
        preds_all = np.concatenate(preds_all)
        labels_all = np.concatenate(labels_all)

        macro_f1 = classification_report(labels_all, preds_all, labels=list(range(NUM_CLASSES)),
                                          output_dict=True, zero_division=0)["macro avg"]["f1-score"]
        flag = "  <-- checkpoint's OWN fold" if ckpt_i == test_j else ""
        print(f"    evaluated on test_set_{test_j}: macro-F1={macro_f1*100:.2f}%{flag}")

    del model
    gc.collect()
    torch.cuda.empty_cache()
    print()


Cross-fold contamination check: does checkpoint i secretly perform well on OTHER folds' test sets too?



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- Checkpoint 1 ---
    evaluated on test_set_1: macro-F1=78.34%  <-- checkpoint's OWN fold
    evaluated on test_set_2: macro-F1=91.79%
    evaluated on test_set_3: macro-F1=82.78%
    evaluated on test_set_4: macro-F1=94.23%



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- Checkpoint 2 ---
    evaluated on test_set_1: macro-F1=90.49%
    evaluated on test_set_2: macro-F1=90.68%  <-- checkpoint's OWN fold
    evaluated on test_set_3: macro-F1=85.64%
    evaluated on test_set_4: macro-F1=76.51%



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- Checkpoint 3 ---
    evaluated on test_set_1: macro-F1=86.59%
    evaluated on test_set_2: macro-F1=92.14%
    evaluated on test_set_3: macro-F1=95.83%  <-- checkpoint's OWN fold
    evaluated on test_set_4: macro-F1=94.91%



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- Checkpoint 4 ---
    evaluated on test_set_1: macro-F1=87.93%
    evaluated on test_set_2: macro-F1=90.40%
    evaluated on test_set_3: macro-F1=94.01%
    evaluated on test_set_4: macro-F1=95.06%  <-- checkpoint's OWN fold



In [26]:
import difflib
import re
from itertools import permutations
from pathlib import Path

import pandas as pd


SPLITS_DIR = "/kaggle/input/datasets/zaimast7454/exact-splits/exact_splits"
NUM_FOLDS = 4
SAMPLES_PER_PAIR = 3      
NEAR_DUP_THRESHOLD = 0.90
RUN_NEAR_DUP = True      


def normalize(text):
    return re.sub(r"\s+", " ", text).strip().lower()


def load_fold(splits_dir, fold_idx, split):
    df = pd.read_csv(Path(splits_dir) / f"{split}_set_{fold_idx}.csv")
    df["full_code"] = df["full_code"].fillna("").astype(str)
    df["_norm"] = df["full_code"].map(normalize)
    return df


def exact_and_normalized_hits(train_df, test_df):
    train_exact = {}
    train_norm = {}
    for idx, row in train_df.iterrows():
        train_exact.setdefault(row["full_code"], []).append(idx)
        train_norm.setdefault(row["_norm"], []).append(idx)

    exact_hits, norm_only_hits = [], []
    for t_idx, t_row in test_df.iterrows():
        code = t_row["full_code"]
        if code in train_exact:
            exact_hits.append((t_idx, train_exact[code][0]))
        elif t_row["_norm"] in train_norm:
            norm_only_hits.append((t_idx, train_norm[t_row["_norm"]][0]))
    return exact_hits, norm_only_hits


def near_dup_hits(train_df, test_df, already_matched_test_idx, threshold, max_candidates_per_row=40):
    """Fuzzy pass restricted to same category + similar length, skipping rows
    already caught as exact/normalized matches. Sampled, not exhaustive."""
    train_by_cat = {}
    for idx, row in train_df.iterrows():
        train_by_cat.setdefault(row["category"], []).append((idx, row["full_code"], len(row["full_code"])))

    hits = []
    for t_idx, t_row in test_df.iterrows():
        if t_idx in already_matched_test_idx:
            continue
        code, length, cat = t_row["full_code"], len(t_row["full_code"]), t_row["category"]
        candidates = train_by_cat.get(cat, [])
        candidates = [c for c in candidates if abs(c[2] - length) <= 0.15 * max(length, 1)]
        candidates = candidates[:max_candidates_per_row]

        best_ratio, best_train_idx = 0.0, None
        for train_idx, train_code, _ in candidates:
            ratio = difflib.SequenceMatcher(None, code, train_code).ratio()
            if ratio > best_ratio:
                best_ratio, best_train_idx = ratio, train_idx
        if best_ratio >= threshold:
            hits.append((t_idx, best_train_idx, best_ratio))
    return hits


def snippet(text, n=160):
    text = re.sub(r"\s+", " ", text).strip()
    return text[:n] + ("..." if len(text) > n else "")


def report_pair(fold_i, fold_j, train_df, test_df, samples_per_pair, near_dup_threshold, run_near_dup):
    exact_hits, norm_only_hits = exact_and_normalized_hits(train_df, test_df)
    total = len(test_df)
    print(f"\ntrain_{fold_i} vs test_{fold_j}: "
          f"exact={len(exact_hits)}/{total} ({len(exact_hits)/total*100:.1f}%), "
          f"normalized-only={len(norm_only_hits)}/{total} ({len(norm_only_hits)/total*100:.1f}%)")

    for tag, hits in [("EXACT", exact_hits), ("NORMALIZED", norm_only_hits)]:
        for t_idx, tr_idx in hits[:samples_per_pair]:
            t_row, tr_row = test_df.loc[t_idx], train_df.loc[tr_idx]
            print(f"  [{tag}] test_{fold_j} project={t_row['project']!r} test={t_row.get('test_name','?')!r}  "
                  f"<->  train_{fold_i} project={tr_row['project']!r} test={tr_row.get('test_name','?')!r}")
            print(f"        code: {snippet(t_row['full_code'])}")

    if run_near_dup:
        already = {t for t, _ in exact_hits} | {t for t, _ in norm_only_hits}
        near_hits = near_dup_hits(train_df, test_df, already, near_dup_threshold)
        print(f"  near-duplicate (ratio>={near_dup_threshold}, not exact/normalized): "
              f"{len(near_hits)}/{total} ({len(near_hits)/total*100:.1f}%)")
        for t_idx, tr_idx, ratio in near_hits[:samples_per_pair]:
            t_row, tr_row = test_df.loc[t_idx], train_df.loc[tr_idx]
            print(f"  [NEAR ratio={ratio:.2f}] test_{fold_j} project={t_row['project']!r}  "
                  f"<->  train_{fold_i} project={tr_row['project']!r}")
            print(f"        test : {snippet(t_row['full_code'])}")
            print(f"        train: {snippet(tr_row['full_code'])}")


folds_train = {i: load_fold(SPLITS_DIR, i, "train") for i in range(1, NUM_FOLDS + 1)}
folds_test = {i: load_fold(SPLITS_DIR, i, "test") for i in range(1, NUM_FOLDS + 1)}

print("=== ON-DIAGONAL (train_i vs test_i) -- this is what actually backs each fold's reported F1 ===")
for i in range(1, NUM_FOLDS + 1):
    report_pair(i, i, folds_train[i], folds_test[i], SAMPLES_PER_PAIR, NEAR_DUP_THRESHOLD, RUN_NEAR_DUP)

print("\n\n=== CROSS-FOLD (train_i vs test_j, i != j) -- explains cross-checkpoint contamination, NOT Table 2 itself ===")
for i, j in permutations(range(1, NUM_FOLDS + 1), 2):
    report_pair(i, j, folds_train[i], folds_test[j], SAMPLES_PER_PAIR, NEAR_DUP_THRESHOLD, RUN_NEAR_DUP)

=== ON-DIAGONAL (train_i vs test_i) -- this is what actually backs each fold's reported F1 ===

train_1 vs test_1: exact=2/2432 (0.1%), normalized-only=0/2432 (0.0%)
  [EXACT] test_1 project='strapdata_elassandra' test='PreBuiltTransportClientTests.testInstallPluginTwice'  <->  train_1 project='opensearch-project_OpenSearch' test='PreBuiltTransportClientTests.testInstallPluginTwice'
        code: @Test public void testInstallPluginTwice() { for (Class<? extends Plugin> plugin : Arrays.asList(ParentJoinPlugin.class, ReindexPlugin.class, PercolatorPlugin.c...
  [EXACT] test_1 project='strapdata_elassandra' test='SuiteScopeClusterIT.testReproducible'  <->  train_1 project='opensearch-project_OpenSearch' test='SuiteScopeClusterIT.testReproducible'
        code: @Test public void testReproducible() throws IOException { if (ITER++ == 0) { CLUSTER_SEED = cluster().seed(); for (int i = 0; i < SEQUENCE.length; i++) { SEQUEN...
  near-duplicate (ratio>=0.9, not exact/normalized): 1/2432 (0.0%)
 

In [27]:
rows = []
for i in range(1, NUM_FOLDS + 1):
    row = {}
    for j in range(1, NUM_FOLDS + 1):
        exact_hits, norm_only_hits = exact_and_normalized_hits(folds_train[i], folds_test[j])
        row[f"test_{j}"] = len(exact_hits) + len(norm_only_hits)
    rows.append(row)

overlap_counts = pd.DataFrame(rows, index=[f"train_{i}" for i in range(1, NUM_FOLDS + 1)])

test_sizes = {j: len(folds_test[j]) for j in range(1, NUM_FOLDS + 1)}
overlap_pct = overlap_counts.copy()
for j in range(1, NUM_FOLDS + 1):
    overlap_pct[f"test_{j}"] = (overlap_counts[f"test_{j}"] / test_sizes[j] * 100).round(1)

print("Common instances (exact + normalized duplicates), count:")
display(overlap_counts)

print("\nSame, as % of that test fold's size:")
display(overlap_pct)

Common instances (exact + normalized duplicates), count:


,test_1,test_2,test_3,test_4
train_1,2,1549,1794,1612
train_2,1942,7,1789,1646
train_3,1957,1547,1,1616
train_4,1963,1546,1764,0



Same, as % of that test fold's size:


,test_1,test_2,test_3,test_4
train_1,0.1,81.2,81.0,79.8
train_2,79.9,0.4,80.7,81.5
train_3,80.5,81.1,0.0,80.0
train_4,80.7,81.1,79.6,0.0


## Method 2




In [15]:
import gc, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, SequentialSampler, TensorDataset
from transformers import AutoConfig, AutoModel, AutoTokenizer

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch: 2.10.0+cu128
CUDA available: True
Device: cuda


In [16]:
EXACT_SPLITS_DIR = "/kaggle/input/datasets/zaimast7454/exact-splits/exact_splits"
PRETRAINED_WEIGHTS_DIR = "/kaggle/input/datasets/zaimast7454/projectfolds"
CHECKPOINT_TEMPLATE = "per_project_model_weights_on__dataset_project_group_{fold}.pt"

OUTPUT_DIR = "/kaggle/working/flakylens_rq1_repro"
BATCH_SIZE = 8
NUM_FOLDS = 4

MODEL_NAME = "microsoft/codebert-base"
MAX_LENGTH = 512
NUM_CLASSES = 6
DROPOUT = 0.30

LABEL_NAMES = {
    0: "Async Wait",
    1: "Concurrency",
    2: "Time",
    3: "Unordered Collections",
    4: "Test Order Dependency",
    5: "Non-flaky",
}
TABLE2_COLUMN_ORDER = [0, 1, 2, 3, 4, 5]


PAPER_TABLE2_FLAKYLENS = {
    "Async Wait": 58.37,
    "Concurrency": 35.92,
    "Time": 72.73,
    "Unordered Collections": 73.63,
    "Test Order Dependency": 64.35,
    "Non-flaky": 100.00,
    "Macro Avg": 65.79,
}

PAPER_TEST_SET_SUPPORT_TOTALS = {1: 2229, 2: 2181, 3: 2199, 4: 1965}

In [17]:
class BERT_Arch(torch.nn.Module):
    def __init__(self, auto_model, num_classes=NUM_CLASSES):
        super().__init__()
        self.bert = auto_model
        self.dropout = torch.nn.Dropout(DROPOUT)
        self.relu = torch.nn.ReLU()
        self.fc1 = torch.nn.Linear(768, 512)
        self.fc2 = torch.nn.Linear(512, num_classes)
        self.log_softmax = torch.nn.LogSoftmax(dim=-1)

    def forward(self, input_ids, attention_mask):
        input_ids = input_ids.long()
        attention_mask = attention_mask.long()
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        pooled = outputs[1]
        x = self.fc1(pooled)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return self.log_softmax(x)


def build_fresh_model():
    config = AutoConfig.from_pretrained(MODEL_NAME, return_dict=False, output_hidden_states=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    auto_model = AutoModel.from_pretrained(MODEL_NAME, config=config)
    model = BERT_Arch(auto_model, NUM_CLASSES).to(device)
    return model, tokenizer


def load_fold_checkpoint(checkpoint_path):
    model, tokenizer = build_fresh_model()
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model, tokenizer

In [18]:
def load_exact_fold_split(fold_idx, splits_dir):
    splits_dir = Path(splits_dir)
    train_df = pd.read_csv(splits_dir / f"train_set_{fold_idx}.csv")
    test_df = pd.read_csv(splits_dir / f"test_set_{fold_idx}.csv")
    for d in (train_df, test_df):
        d["project"] = d["project"].astype(str)
        d["full_code"] = d["full_code"].fillna("").astype(str)
        d["category"] = pd.to_numeric(d["category"], errors="raise").astype(int)
    overlap = set(train_df["project"]) & set(test_df["project"])
    if overlap:
        raise ValueError(f"fold {fold_idx}: {len(overlap)} project(s) leak between train and test: {sorted(overlap)[:5]}...")
    return train_df, test_df

In [19]:
@torch.no_grad()
def predict(model, tokenizer, texts, batch_size=BATCH_SIZE):
    tokens = tokenizer(
        texts.tolist(), max_length=MAX_LENGTH, padding="max_length", truncation=True, return_tensors="pt"
    )
    dataset = TensorDataset(tokens["input_ids"], tokens["attention_mask"])
    loader = DataLoader(dataset, sampler=SequentialSampler(dataset), batch_size=batch_size)
    model.eval()
    preds = []
    for input_ids, attention_mask in loader:
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        outputs = model(input_ids, attention_mask)
        preds.append(torch.argmax(outputs, dim=1).cpu().numpy())
    return np.concatenate(preds)


def evaluate_fold(fold_idx, checkpoint_path, splits_dir, batch_size=BATCH_SIZE):
    print(f"\n{'=' * 70}\nFOLD {fold_idx}\n{'=' * 70}")
    _, test_df = load_exact_fold_split(fold_idx, splits_dir)
    observed_support = len(test_df)
    expected_support = PAPER_TEST_SET_SUPPORT_TOTALS.get(fold_idx)
    if expected_support is not None and observed_support != expected_support:
        print(
            f"  NOTE: this fold's test set has {observed_support} rows; the published run's "
            f"fold {fold_idx} test set had {expected_support}. Different project grouping "
            f"(see the caveat above) -- do not expect a bit-exact match to Table 2 from this alone."
        )

    model, tokenizer = load_fold_checkpoint(checkpoint_path)
    preds = predict(model, tokenizer, test_df["full_code"], batch_size=batch_size)
    labels = test_df["category"].to_numpy()

    report = classification_report(
        labels, preds, labels=TABLE2_COLUMN_ORDER,
        target_names=[LABEL_NAMES[i] for i in TABLE2_COLUMN_ORDER],
        output_dict=True, zero_division=0,
    )
    fold_macro_f1 = report["macro avg"]["f1-score"]
    print(f"  test support: {observed_support}  |  fold macro-F1: {fold_macro_f1 * 100:.2f}%")

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "fold": fold_idx,
        "report": report,
        "preds": preds,
        "labels": labels,
        "confusion_matrix": confusion_matrix(labels, preds, labels=TABLE2_COLUMN_ORDER),
    }

In [20]:
fold_results = []
for fold_idx in range(1, NUM_FOLDS + 1):
    ckpt_path = Path(PRETRAINED_WEIGHTS_DIR) / CHECKPOINT_TEMPLATE.format(fold=fold_idx)
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    fold_results.append(evaluate_fold(fold_idx, ckpt_path, EXACT_SPLITS_DIR))

print(f"\n{len(fold_results)}/{NUM_FOLDS} folds evaluated.")


FOLD 1
  NOTE: this fold's test set has 2432 rows; the published run's fold 1 test set had 2229. Different project grouping (see the caveat above) -- do not expect a bit-exact match to Table 2 from this alone.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  test support: 2432  |  fold macro-F1: 78.34%

FOLD 2
  NOTE: this fold's test set has 1907 rows; the published run's fold 2 test set had 2181. Different project grouping (see the caveat above) -- do not expect a bit-exact match to Table 2 from this alone.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  test support: 1907  |  fold macro-F1: 90.68%

FOLD 3
  NOTE: this fold's test set has 2216 rows; the published run's fold 3 test set had 2199. Different project grouping (see the caveat above) -- do not expect a bit-exact match to Table 2 from this alone.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  test support: 2216  |  fold macro-F1: 95.83%

FOLD 4
  NOTE: this fold's test set has 2019 rows; the published run's fold 4 test set had 1965. Different project grouping (see the caveat above) -- do not expect a bit-exact match to Table 2 from this alone.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  test support: 2019  |  fold macro-F1: 95.06%

4/4 folds evaluated.


In [21]:
def aggregate_like_the_paper(fold_results):
    per_category = {}
    for cat_idx in TABLE2_COLUMN_ORDER:
        name = LABEL_NAMES[cat_idx]
        f1_times_support = 0.0
        total_support = 0.0
        for r in fold_results:
            row = r["report"][name]
            f1_times_support += row["f1-score"] * row["support"]
            total_support += row["support"]
        per_category[name] = 100.0 * f1_times_support / total_support if total_support else 0.0

    fold_macro_f1s = [r["report"]["macro avg"]["f1-score"] for r in fold_results]
    macro_avg = 100.0 * float(np.mean(fold_macro_f1s))

    return per_category, macro_avg, fold_macro_f1s


per_category, macro_avg, fold_macro_f1s = aggregate_like_the_paper(fold_results)
print("Per-fold macro-F1:", [f"{v * 100:.2f}%" for v in fold_macro_f1s])

Per-fold macro-F1: ['78.34%', '90.68%', '95.83%', '95.06%']


In [22]:
header = ["Async.", "Conc.", "Time", "UC", "OD", "Non-flaky", "Macro Avg."]
row_yours = [per_category[LABEL_NAMES[i]] for i in TABLE2_COLUMN_ORDER] + [macro_avg]
row_paper = [PAPER_TABLE2_FLAKYLENS[LABEL_NAMES[i]] for i in TABLE2_COLUMN_ORDER] + [PAPER_TABLE2_FLAKYLENS["Macro Avg"]]
summary_df = pd.DataFrame([row_yours, row_paper], columns=header, index=["Yours", "Paper (Table 2)"]).round(2)
display(summary_df)

,Async.,Conc.,Time,UC,OD,Non-flaky,Macro Avg.
Yours,86.92,82.79,89.29,90.63,85.12,100.0,89.98
Paper (Table 2),58.37,35.92,72.73,73.63,64.35,100.0,65.79


In [23]:
per_fold_rows = []
for r in fold_results:
    row = {"fold": r["fold"], "macro_f1": r["report"]["macro avg"]["f1-score"] * 100}
    for i in TABLE2_COLUMN_ORDER:
        row[LABEL_NAMES[i]] = r["report"][LABEL_NAMES[i]]["f1-score"] * 100
    per_fold_rows.append(row)
per_fold_df = pd.DataFrame(per_fold_rows).set_index("fold").round(2)
display(per_fold_df)

,macro_f1,Async Wait,Concurrency,Time,Unordered Collections,Test Order Dependency,Non-flaky
fold,,,,,,,
1,78.34,83.12,80.00,66.67,82.35,57.89,100.0
2,90.68,86.96,57.14,100.00,100.00,100.00,100.0
3,95.83,96.00,100.00,93.33,92.31,93.33,100.0
4,95.06,88.24,90.91,100.00,95.65,95.56,100.0


In [28]:
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(out / "table2_comparison.csv")
per_fold_df.to_csv(out / "per_fold_f1.csv")
with open(out / "config.json", "w") as f:
    json.dump({"batch_size": BATCH_SIZE, "splits_dir": EXACT_SPLITS_DIR, "weights_dir": PRETRAINED_WEIGHTS_DIR}, f, indent=2)
print("Saved results to", out)
for p in sorted(out.iterdir()):
    print(" -", p.name)

Saved results to /kaggle/working/flakylens_rq1_repro
 - config.json
 - per_fold_f1.csv
 - table2_comparison.csv
